# Plan de accion de Fase 2 (Protocolo experimental)

Este notebook se dedica exclusivamente a cerrar los pendientes de la Fase 2 con trazabilidad reproducible.

Objetivo: convertir la documentacion del protocolo en artefactos ejecutados y exportados (splits, CV, validacion de candidatas y decisiones finales).

## 1. Alcance y entregables

Entregables minimos que deben quedar generados al cerrar esta fase:

1. Split externo estratificado persistido (train_val / test).
2. Configuracion de validacion interna estratificada y semilla comun.
3. Resultados CV por candidata (base vs extendido) para accuracy y recall.
4. Decision final de candidatas (aceptar/rechazar) segun reglas del protocolo.
5. Tabla de trazabilidad final exportada para memoria.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold

BASE_DIR = Path('..')
DATA_PATH = BASE_DIR / 'data' / 'raw' / 'BBDD_ML_TAREA.csv'
TABLES_PATH = BASE_DIR / 'outputs' / 'tables'
TABLES_PATH.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5

## 2. Estado actual automatico

Esta celda inspecciona si ya existen los artefactos clave de Fase 2 y marca su estado actual.

In [2]:
required_artifacts = [
    'fase2_split_indices.csv',
    'fase2_split_config.json',
    'fase2_feature_eval_template.csv',
    'fase2_feature_eval_decisions.csv',
    'fase2_feature_eval_metrics.csv',
    'fase2_action_plan_status.csv'
]

status_rows = []
for f in required_artifacts:
    p = TABLES_PATH / f
    status_rows.append({
        'artifact': f,
        'exists': p.exists(),
        'path': str(p)
    })

artifact_status = pd.DataFrame(status_rows)
display(artifact_status)

,artifact,exists,path
0,fase2_split_indices.csv,True,..\outputs\tables\fase2_split_indices.csv
1,fase2_split_config.json,True,..\outputs\tables\fase2_split_config.json
2,fase2_feature_eval_template.csv,True,..\outputs\tables\fase2_feature_eval_template.csv
3,fase2_feature_eval_decisions.csv,True,..\outputs\tables\fase2_feature_eval_decisions...
4,fase2_feature_eval_metrics.csv,False,..\outputs\tables\fase2_feature_eval_metrics.csv
5,fase2_action_plan_status.csv,True,..\outputs\tables\fase2_action_plan_status.csv


## 3. Plan de accion operativo

Checklist accionable de pendientes. Se actualiza automaticamente con cada ejecucion.

In [3]:
action_plan = pd.DataFrame([
    {'step': 'Crear split externo estratificado y guardarlo', 'owner': 'notebook', 'status': 'pendiente'},
    {'step': 'Guardar configuracion formal del split/CV', 'owner': 'notebook', 'status': 'pendiente'},
    {'step': 'Calcular metricas CV base vs extendido por candidata', 'owner': 'notebook', 'status': 'pendiente'},
    {'step': 'Aplicar regla de decision y exportar aceptar/rechazar', 'owner': 'notebook', 'status': 'pendiente'},
    {'step': 'Generar resumen final para memoria', 'owner': 'notebook', 'status': 'pendiente'}
])

display(action_plan)

,step,owner,status
0,Crear split externo estratificado y guardarlo,notebook,pendiente
1,Guardar configuracion formal del split/CV,notebook,pendiente
2,Calcular metricas CV base vs extendido por can...,notebook,pendiente
3,Aplicar regla de decision y exportar aceptar/r...,notebook,pendiente
4,Generar resumen final para memoria,notebook,pendiente


## 4. Paso 1 - Split externo estratificado

Se construye sobre el conjunto base de modelado (deduplicado y sin variables excluidas por diseno), y se guardan indices para reutilizar exactamente el mismo reparto en todo el proyecto.

In [4]:
df_raw = pd.read_csv(DATA_PATH, na_values=['NA'])
df_work = df_raw.drop_duplicates().copy()
cols_to_drop = ['V4', 'V10', 'V13', 'V16', 'V19']
df_model_base = df_work.drop(columns=cols_to_drop).copy()

X = df_model_base.drop(columns=['Y'])
y = df_model_base['Y']

idx = np.arange(len(df_model_base))
idx_train_val, idx_test = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

split_df = pd.DataFrame({
    'row_idx': idx,
    'set': np.where(np.isin(idx, idx_test), 'test', 'train_val')
})
split_df.to_csv(TABLES_PATH / 'fase2_split_indices.csv', index=False)

split_config = {
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'n_splits_cv': N_SPLITS,
    'dataset_shape': list(df_model_base.shape)
}
with open(TABLES_PATH / 'fase2_split_config.json', 'w', encoding='utf-8') as f:
    json.dump(split_config, f, ensure_ascii=False, indent=2)

print('train_val:', (split_df['set'] == 'train_val').sum())
print('test:', (split_df['set'] == 'test').sum())
display(split_df['set'].value_counts())

train_val: 2830
test: 708


set
train_val    2830
test          708
Name: count, dtype: int64

## 5. Paso 2 - Configuracion CV interna

Se registra de forma explicita la configuracion de validacion cruzada estratificada que se usara para evaluar candidatas.

In [5]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_info = pd.DataFrame({
    'parameter': ['n_splits', 'shuffle', 'random_state'],
    'value': [N_SPLITS, True, RANDOM_STATE]
})
display(cv_info)

,parameter,value
0,n_splits,5
1,shuffle,True
2,random_state,42


## 6. Paso 3 y 4 - Pendientes de evaluacion de candidatas

Este notebook deja el plan y la infraestructura listos. El calculo de metricas CV base vs extendido por candidata se implementara en el notebook de evaluacion/modelado para no duplicar logica experimental.

Mientras tanto, se guarda un estado consolidado de avance.

In [6]:
status_map = {
    'Crear split externo estratificado y guardarlo': (TABLES_PATH / 'fase2_split_indices.csv').exists(),
    'Guardar configuracion formal del split/CV': (TABLES_PATH / 'fase2_split_config.json').exists(),
    'Calcular metricas CV base vs extendido por candidata': (TABLES_PATH / 'fase2_feature_eval_metrics.csv').exists(),
    'Aplicar regla de decision y exportar aceptar/rechazar': (TABLES_PATH / 'fase2_feature_eval_decisions.csv').exists(),
    'Generar resumen final para memoria': (TABLES_PATH / 'fase2_feature_eval_summary_counts.csv').exists()
}

action_plan['status'] = action_plan['step'].map(lambda s: 'hecho' if status_map.get(s, False) else 'pendiente')
display(action_plan)
action_plan.to_csv(TABLES_PATH / 'fase2_action_plan_status.csv', index=False)
print('Estado exportado en:', (TABLES_PATH / 'fase2_action_plan_status.csv').resolve())


,step,owner,status
0,Crear split externo estratificado y guardarlo,notebook,hecho
1,Guardar configuracion formal del split/CV,notebook,hecho
2,Calcular metricas CV base vs extendido por can...,notebook,pendiente
3,Aplicar regla de decision y exportar aceptar/r...,notebook,hecho
4,Generar resumen final para memoria,notebook,pendiente


Estado exportado en: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea\outputs\tables\fase2_action_plan_status.csv


## 6. Paso 3 y 4 - Evaluacion CV de candidatas y decision automatica

Se compara modelo base vs extendido por candidata usando exclusivamente `train_val` y validacion cruzada estratificada.
Se exportan metricas y decisiones para trazabilidad en memoria.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RECALL_TOLERANCE_ACC = 0.02
PRECISION_TOLERANCE_REC = 0.02
PRAUC_TOLERANCE_REC = 0.02
STD_INCREASE_LIMIT = 0.20

df_raw = pd.read_csv(DATA_PATH, na_values=['NA'])
df_work = df_raw.drop_duplicates().copy()
cols_to_drop = ['V4', 'V10', 'V13', 'V16', 'V19']
df_model_base = df_work.drop(columns=cols_to_drop).copy()

split_df = pd.read_csv(TABLES_PATH / 'fase2_split_indices.csv')
train_idx = split_df.loc[split_df['set'] == 'train_val', 'row_idx'].to_numpy()
df_train_val = df_model_base.iloc[train_idx].reset_index(drop=True)

X_base = df_train_val.drop(columns=['Y']).copy()
y_train_val = df_train_val['Y'].copy()

feature_candidates = pd.read_csv(TABLES_PATH / 'fase2_feature_candidates.csv')
candidates = feature_candidates['feature'].tolist()

def apply_candidate(df_features, candidate_name):
    df = df_features.copy()
    total_minutes = df['V8'] + df['V11'] + df['V14'] + df['V17']
    total_calls = df['V9'] + df['V12'] + df['V15'] + df['V18']

    if candidate_name == 'total_minutes':
        df['total_minutes'] = total_minutes
    elif candidate_name == 'total_calls':
        df['total_calls'] = total_calls
    elif candidate_name == 'minutes_per_call':
        df['minutes_per_call'] = np.where(total_calls > 0, total_minutes / total_calls, np.nan)
    elif candidate_name == 'international_share_minutes':
        df['international_share_minutes'] = np.where(total_minutes > 0, df['V17'] / total_minutes, 0.0)
    elif candidate_name == 'log1p_v2':
        df['log1p_v2'] = np.log1p(df['V2'].clip(lower=0))
    elif candidate_name == 'log1p_total_minutes':
        df['log1p_total_minutes'] = np.log1p(total_minutes.clip(lower=0))
    elif candidate_name == 'customer_service_intensity':
        df['customer_service_intensity'] = df['V20']
    else:
        raise ValueError(f'Candidata no soportada: {candidate_name}')
    return df

def build_preprocessor(X):
    nominal_cols = [c for c in ['V1', 'V3'] if c in X.columns]
    numeric_cols = [c for c in X.columns if c not in nominal_cols]

    nominal_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    numeric_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    return ColumnTransformer(
        transformers=[
            ('nominal', nominal_pipe, nominal_cols),
            ('numeric', numeric_pipe, numeric_cols),
        ],
        remainder='drop'
    )

def evaluate_cv(X):
    pipe = Pipeline(steps=[
        ('prep', build_preprocessor(X)),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, solver='lbfgs'))
    ])

    scoring = {
        'accuracy': 'accuracy',
        'recall': 'recall',
        'precision': 'precision',
        'prauc': 'average_precision',
    }

    res = cross_validate(pipe, X, y_train_val, cv=cv, scoring=scoring, n_jobs=None, error_score='raise')
    return {
        'acc_mean': float(np.mean(res['test_accuracy'])),
        'recall_mean': float(np.mean(res['test_recall'])),
        'precision_mean': float(np.mean(res['test_precision'])),
        'prauc_mean': float(np.mean(res['test_prauc'])),
        'acc_std': float(np.std(res['test_accuracy'], ddof=1)),
        'recall_std': float(np.std(res['test_recall'], ddof=1)),
    }

def decide_feature_candidate(row):
    std_ratio = (row['primary_std_ext'] / row['primary_std_base']) if row['primary_std_base'] > 0 else np.inf
    unstable = std_ratio > (1 + STD_INCREASE_LIMIT)

    if row['objective'] == 'accuracy':
        primary_ok = row['acc_ext_mean'] > row['acc_base_mean']
        guardrail_ok = (row['recall_base_mean'] - row['recall_ext_mean']) <= RECALL_TOLERANCE_ACC
    else:
        primary_ok = row['recall_ext_mean'] > row['recall_base_mean']
        guardrail_ok = (
            (row['precision_base_mean'] - row['precision_ext_mean']) <= PRECISION_TOLERANCE_REC
            and (row['prauc_base_mean'] - row['prauc_ext_mean']) <= PRAUC_TOLERANCE_REC
        )

    if primary_ok and guardrail_ok and not unstable:
        return pd.Series({'decision': 'aceptar', 'reason': 'mejora primaria, respeta guardrails y mantiene estabilidad'})

    reason_parts = []
    if not primary_ok:
        reason_parts.append('no mejora metrica primaria')
    if not guardrail_ok:
        reason_parts.append('viola guardrails secundarios')
    if unstable:
        reason_parts.append('incrementa inestabilidad entre pliegues')
    return pd.Series({'decision': 'rechazar', 'reason': '; '.join(reason_parts)})

base_metrics = evaluate_cv(X_base)
rows = []
for cand in candidates:
    X_ext = apply_candidate(X_base, cand)
    ext_metrics = evaluate_cv(X_ext)

    for objective in ['accuracy', 'recall']:
        primary_std_base = base_metrics['acc_std'] if objective == 'accuracy' else base_metrics['recall_std']
        primary_std_ext = ext_metrics['acc_std'] if objective == 'accuracy' else ext_metrics['recall_std']

        rows.append({
            'candidate': cand,
            'objective': objective,
            'acc_base_mean': base_metrics['acc_mean'],
            'acc_ext_mean': ext_metrics['acc_mean'],
            'recall_base_mean': base_metrics['recall_mean'],
            'recall_ext_mean': ext_metrics['recall_mean'],
            'precision_base_mean': base_metrics['precision_mean'],
            'precision_ext_mean': ext_metrics['precision_mean'],
            'prauc_base_mean': base_metrics['prauc_mean'],
            'prauc_ext_mean': ext_metrics['prauc_mean'],
            'primary_std_base': primary_std_base,
            'primary_std_ext': primary_std_ext,
        })

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(TABLES_PATH / 'fase2_feature_eval_metrics.csv', index=False)

decisions_df = metrics_df.join(metrics_df.apply(decide_feature_candidate, axis=1))
decisions_df.to_csv(TABLES_PATH / 'fase2_feature_eval_decisions.csv', index=False)

display(decisions_df[['candidate', 'objective', 'decision', 'reason']])
print('Metricas exportadas en:', (TABLES_PATH / 'fase2_feature_eval_metrics.csv').resolve())
print('Decisiones exportadas en:', (TABLES_PATH / 'fase2_feature_eval_decisions.csv').resolve())


,candidate,objective,decision,reason
0,total_minutes,accuracy,aceptar,"mejora primaria, respeta guardrails y mantiene..."
1,total_minutes,recall,rechazar,no mejora metrica primaria
2,total_calls,accuracy,aceptar,"mejora primaria, respeta guardrails y mantiene..."
3,total_calls,recall,aceptar,"mejora primaria, respeta guardrails y mantiene..."
4,minutes_per_call,accuracy,aceptar,"mejora primaria, respeta guardrails y mantiene..."
5,minutes_per_call,recall,aceptar,"mejora primaria, respeta guardrails y mantiene..."
6,international_share_minutes,accuracy,aceptar,"mejora primaria, respeta guardrails y mantiene..."
7,international_share_minutes,recall,aceptar,"mejora primaria, respeta guardrails y mantiene..."
8,log1p_v2,accuracy,rechazar,no mejora metrica primaria
9,log1p_v2,recall,rechazar,no mejora metrica primaria


Metricas exportadas en: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea\outputs\tables\fase2_feature_eval_metrics.csv
Decisiones exportadas en: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea\outputs\tables\fase2_feature_eval_decisions.csv


## 7. Resumen final para memoria

Se consolida un resumen compacto con recuento de candidatas aceptadas/rechazadas por objetivo y el detalle mínimo para redacción.

In [8]:
decisions_df = pd.read_csv(TABLES_PATH / 'fase2_feature_eval_decisions.csv')

summary_counts = (
    decisions_df.groupby(['objective', 'decision'])['candidate']
    .count()
    .rename('n_candidates')
    .reset_index()
)

summary_delta = decisions_df.assign(
    delta_acc=lambda d: d['acc_ext_mean'] - d['acc_base_mean'],
    delta_recall=lambda d: d['recall_ext_mean'] - d['recall_base_mean'],
    delta_precision=lambda d: d['precision_ext_mean'] - d['precision_base_mean'],
    delta_prauc=lambda d: d['prauc_ext_mean'] - d['prauc_base_mean'],
)[[
    'candidate', 'objective', 'decision', 'delta_acc', 'delta_recall', 'delta_precision', 'delta_prauc'
]]

summary_counts.to_csv(TABLES_PATH / 'fase2_feature_eval_summary_counts.csv', index=False)
summary_delta.to_csv(TABLES_PATH / 'fase2_feature_eval_summary_delta.csv', index=False)

display(summary_counts)
display(summary_delta.head(10))
print('Resumen exportado en:', (TABLES_PATH / 'fase2_feature_eval_summary_counts.csv').resolve())


,objective,decision,n_candidates
0,accuracy,aceptar,4
1,accuracy,rechazar,3
2,recall,aceptar,4
3,recall,rechazar,3


,candidate,objective,decision,delta_acc,delta_recall,delta_precision,delta_prauc
0,total_minutes,accuracy,aceptar,0.000353,0.000000,0.001687,-0.000214
1,total_minutes,recall,rechazar,0.000353,0.000000,0.001687,-0.000214
2,total_calls,accuracy,aceptar,0.000353,0.001770,0.000602,0.000659
3,total_calls,recall,aceptar,0.000353,0.001770,0.000602,0.000659
4,minutes_per_call,accuracy,aceptar,0.000353,0.001770,0.001847,0.001666
5,minutes_per_call,recall,aceptar,0.000353,0.001770,0.001847,0.001666
6,international_share_minutes,accuracy,aceptar,0.006714,0.024779,0.027537,0.019233
7,international_share_minutes,recall,aceptar,0.006714,0.024779,0.027537,0.019233
8,log1p_v2,accuracy,rechazar,-0.000707,-0.001770,-0.002821,-0.000921
9,log1p_v2,recall,rechazar,-0.000707,-0.001770,-0.002821,-0.000921


Resumen exportado en: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea\outputs\tables\fase2_feature_eval_summary_counts.csv


## 7. Criterio de salida de Fase 2

La fase se considera cerrada cuando:

- split y configuracion CV esten guardados,
- exista tabla de metricas CV por candidata,
- exista tabla final de decisiones aceptar/rechazar,
- y el estado consolidado marque todo en hecho salvo, como maximo, el resumen narrativo final para memoria.